In [2]:
#load the data

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

loader = TextLoader('speech.txt')
documents = loader.load()


#split the data

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)
docs



c:\Users\Rahul Singh\Desktop\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'speech.txt'}, page_content='artificial intelligence (AI), the ability of a digital computer or computer-controlled robot to perform tasks commonly associated with intelligent beings. The term is frequently applied to the project of developing systems endowed with the intellectual processes characteristic of humans, such as the ability to reason, discover meaning, generalize, or learn from past experience. Since their development in the 1940s, digital computers have been programmed to carry out very complex tasksâ€”such as'),
 Document(metadata={'source': 'speech.txt'}, page_content='to carry out very complex tasksâ€”such as discovering proofs for mathematical theorems or playing chessâ€”with great proficiency. Despite continuing advances in computer processing speed and memory capacity, there are as yet no programs that can match full human flexibility over wider domains or in tasks requiring much everyday knowledge. On the other hand, some programs have 

In [28]:
# embedding the documents
from langchain_ollama import OllamaEmbeddings
embeddings=OllamaEmbeddings(model='nomic-embed-text')
result=embeddings.embed_documents([docs[0].page_content for doc in docs])
result









[[0.022149166,
  0.09767671,
  -0.1507511,
  -0.02778952,
  0.06322958,
  0.059299387,
  0.009163662,
  -0.023164934,
  -0.016504688,
  -0.003923296,
  0.0045407787,
  0.029080553,
  0.09731414,
  0.052422334,
  0.023920301,
  -0.051811434,
  -0.021698732,
  -0.015005105,
  0.011891913,
  0.02908489,
  0.033112932,
  -0.03846639,
  -0.008616291,
  0.030928506,
  0.05212209,
  0.0035223335,
  -0.041283812,
  -0.04837892,
  0.02023381,
  -0.010517213,
  0.021327402,
  -0.035199896,
  0.03363593,
  0.02832666,
  -0.046428617,
  -0.05482579,
  0.076764345,
  0.019688493,
  0.04988882,
  0.039454542,
  -0.03706229,
  0.05426689,
  -0.012410956,
  -0.019171039,
  0.06832578,
  0.036256846,
  0.0018903253,
  -0.046536103,
  0.047692094,
  -0.03785922,
  0.03515653,
  -0.02851236,
  -0.024859032,
  -0.020446567,
  0.11482057,
  0.020479556,
  -0.01735937,
  0.0126368785,
  -0.00076498336,
  -0.030045979,
  0.10312713,
  0.07202654,
  -0.033455912,
  0.057716023,
  0.051546443,
  -0.0223217,
  

In [4]:
#store the embedding in FAISS
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(docs,embeddings)
print("FAISS vectorstore created successfully!")


FAISS vectorstore created successfully!


In [29]:
# create retriever

retriever=vectorstore.as_retriever(search_kwargs={'k':3})
retriever









VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000013E4A4EE300>, search_kwargs={'k': 3})

In [6]:
# test the retriever
query="what is the speech about?"
docs_retrieved=retriever.invoke(query)
print("retriever chunks",docs_retrieved)








retriever chunks [Document(id='d951022b-7bcf-4515-87df-680485d7314e', metadata={'source': 'speech.txt'}, page_content='artificial intelligence (AI), the ability of a digital computer or computer-controlled robot to perform tasks commonly associated with intelligent beings. The term is frequently applied to the project of developing systems endowed with the intellectual processes characteristic of humans, such as the ability to reason, discover meaning, generalize, or learn from past experience. Since their development in the 1940s, digital computers have been programmed to carry out very complex tasksâ€”such as'), Document(id='de8ffcee-712e-4c1d-bcec-2797f7d67256', metadata={'source': 'speech.txt'}, page_content='in executing certain specific tasks, so that artificial intelligence in this limited sense is found in applications as diverse as medical diagnosis, computer search engines, voice or handwriting recognition, and chatbots.'), Document(id='4461766f-67d5-4388-875f-cc4a21e59377', 

In [30]:
#   retrieved chunks

for i,doc in enumerate(docs_retrieved):
    print(f"\n--- Retrieved chunk {i+1} ---")
    print(doc.page_content)
    


--- Retrieved chunk 1 ---
artificial intelligence (AI), the ability of a digital computer or computer-controlled robot to perform tasks commonly associated with intelligent beings. The term is frequently applied to the project of developing systems endowed with the intellectual processes characteristic of humans, such as the ability to reason, discover meaning, generalize, or learn from past experience. Since their development in the 1940s, digital computers have been programmed to carry out very complex tasksâ€”such as

--- Retrieved chunk 2 ---
in executing certain specific tasks, so that artificial intelligence in this limited sense is found in applications as diverse as medical diagnosis, computer search engines, voice or handwriting recognition, and chatbots.

--- Retrieved chunk 3 ---
to carry out very complex tasksâ€”such as discovering proofs for mathematical theorems or playing chessâ€”with great proficiency. Despite continuing advances in computer processing speed and memory

In [31]:
vectorstore.similarity_search(query, k=3)

[Document(id='d951022b-7bcf-4515-87df-680485d7314e', metadata={'source': 'speech.txt'}, page_content='artificial intelligence (AI), the ability of a digital computer or computer-controlled robot to perform tasks commonly associated with intelligent beings. The term is frequently applied to the project of developing systems endowed with the intellectual processes characteristic of humans, such as the ability to reason, discover meaning, generalize, or learn from past experience. Since their development in the 1940s, digital computers have been programmed to carry out very complex tasksâ€”such as'),
 Document(id='de8ffcee-712e-4c1d-bcec-2797f7d67256', metadata={'source': 'speech.txt'}, page_content='in executing certain specific tasks, so that artificial intelligence in this limited sense is found in applications as diverse as medical diagnosis, computer search engines, voice or handwriting recognition, and chatbots.'),
 Document(id='4461766f-67d5-4388-875f-cc4a21e59377', metadata={'sour

In [41]:
# load the Ollama LLM

from langchain_ollama import ChatOllama

llm= ChatOllama(model='gemma:2b')

print('ollamachat loaded succesfully')



ollamachat loaded succesfully


In [42]:
# combined the retreaved chunks

query="what is the speech about?"

result=vectorstore.similarity_search(query,k=3)

# combine their text

context='\n\n'.join(doc.page_content for doc in result)
context



'artificial intelligence (AI), the ability of a digital computer or computer-controlled robot to perform tasks commonly associated with intelligent beings. The term is frequently applied to the project of developing systems endowed with the intellectual processes characteristic of humans, such as the ability to reason, discover meaning, generalize, or learn from past experience. Since their development in the 1940s, digital computers have been programmed to carry out very complex tasksâ€”such as\n\nin executing certain specific tasks, so that artificial intelligence in this limited sense is found in applications as diverse as medical diagnosis, computer search engines, voice or handwriting recognition, and chatbots.\n\nto carry out very complex tasksâ€”such as discovering proofs for mathematical theorems or playing chessâ€”with great proficiency. Despite continuing advances in computer processing speed and memory capacity, there are as yet no programs that can match full human flexibil

In [43]:
# create prompt

from multiprocessing import context


prompt=f""" 
answer the question using the context below.

Context:
{context}

question:
{query}

if the answer is not available in context, say:
"i dont know the based you provided."

Answer:
"""    



In [44]:
# send it to ollama

response=llm.invoke(prompt)
response.content

'The context provides a module named `multiprocessing.context` which is used for implementing multithreading in Python. The module provides a context manager for creating and managing threads, as well as for synchronizing threads and preventing deadlocks.'

In [45]:
llm.invoke(prompt)

AIMessage(content="The context provides information about a module called 'multiprocessing.context'. The speech is about the multiprocessing module and its use in programming.", additional_kwargs={}, response_metadata={'model': 'gemma:2b', 'created_at': '2026-08-08T05:18:02.556416Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24598335600, 'load_duration': 8666416500, 'prompt_eval_count': 100, 'prompt_eval_duration': 552597000, 'eval_count': 28, 'eval_duration': 15304824000, 'logprobs': None, 'model_name': 'gemma:2b', 'model_provider': 'ollama'}, id='lc_run--019fdfce-28b6-7be3-9739-ee043130defc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 100, 'output_tokens': 28, 'total_tokens': 128})

In [46]:
query = "What is the main message of the speech?"

In [51]:
response = llm.invoke("What is application in ai?")

print(response.content)

Sure, here's a comprehensive definition of application in AI:

**Application** is a specific use case for artificial intelligence (AI) that involves a particular set of data, task, and objectives. It is the real-world scenario or problem that AI is designed to address.

**Here are some key characteristics of an application:**

* **Clear problem statement:** It should clearly define the problem that the AI aims to solve.
* **Well-defined data set:** The application typically requires a large and relevant dataset for training and testing.
* **Specific objectives:** It should have well-defined objectives that need to be accomplished by the AI.
* **Targeted AI techniques:** AI techniques such as machine learning, deep learning, or natural language processing are typically employed to solve the problem.
* **Output or product:** It produces a meaningful output or product, such as predictions, insights, or a new artifact.

**Examples of applications include:**

* **Image recognition:** Traini